# Train the edge-aware mesh refiner (Colab Free)
Run all mặc định dùng bốn sample synthetic, batch size 1, mixed precision khi có CUDA, cache base mesh và quick overfit. Trong Colab, Drive được mount tự động và checkpoint được resume sau ngắt phiên.

In [ ]:
import importlib.util, os, shutil, subprocess, sys, time
from pathlib import Path
STARTED = time.perf_counter()
cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [cwd, cwd / 'image-to-3d-shapenet', *cwd.parents] if (p / 'configs' / 'quick_test.yaml').is_file()), None)
if PROJECT_ROOT is None:
    clone_target = Path('/content/image-to-3d-shapenet')
    try:
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/dinhcongtiendat2003/image-to-3d-shapenet.git', str(clone_target)], check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError('Không tìm thấy hoặc clone được repository từ GitHub.') from exc
    PROJECT_ROOT = clone_target
try:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_ROOT / 'requirements-colab.txt')], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT_ROOT)], check=True)
except subprocess.CalledProcessError as exc:
    raise RuntimeError('Cài dependency thất bại; Restart runtime rồi Run all lại.') from exc
import torch
CUDA = torch.cuda.is_available()
HF_TOKEN_AVAILABLE = bool(os.getenv('HF_TOKEN'))
try:
    from google.colab import userdata
    HF_TOKEN_AVAILABLE = HF_TOKEN_AVAILABLE or bool(userdata.get('HF_TOKEN'))
except Exception:
    pass
print({'project': str(PROJECT_ROOT), 'cuda': CUDA, 'gpu': torch.cuda.get_device_name(0) if CUDA else None, 'vram_gb': round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2) if CUDA else 0.0, 'mixed_precision': CUDA, 'hf_token_available': HF_TOKEN_AVAILABLE})
def run_checked(command, label):
    try:
        subprocess.run(command, cwd=PROJECT_ROOT, check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(f'{label} thất bại; xem log và kiểm tra RAM/disk/checkpoint.') from exc

In [ ]:
USE_GOOGLE_DRIVE = os.getenv('USE_GOOGLE_DRIVE', 'auto') == '1' or ('google.colab' in sys.modules and os.getenv('USE_GOOGLE_DRIVE', 'auto') != '0')
CHECKPOINT_DIR = PROJECT_ROOT / 'outputs' / 'quick_test' / 'notebook_checkpoints'
if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        mount = os.getenv('GOOGLE_DRIVE_MOUNT', '/content/drive')
        drive.mount(mount)
        CHECKPOINT_DIR = Path(mount) / 'MyDrive' / 'image-to-3d-shapenet' / 'checkpoints'
    except Exception as exc:
        CHECKPOINT_DIR = PROJECT_ROOT / 'outputs' / 'quick_test' / 'notebook_checkpoints'
        print(f'Google Drive mount không khả dụng ({type(exc).__name__}); dùng checkpoint local.')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print('checkpoint_dir =', CHECKPOINT_DIR)

In [ ]:
CONFIG = PROJECT_ROOT / 'configs' / 'quick_test.yaml'
run_checked([sys.executable, str(PROJECT_ROOT / 'scripts' / 'prepare_dataset.py'), '--config', str(CONFIG)], 'prepare dataset')
run_checked([sys.executable, str(PROJECT_ROOT / 'scripts' / 'render_dataset.py'), '--config', str(CONFIG)], 'render dataset')
run_checked([sys.executable, str(PROJECT_ROOT / 'scripts' / 'generate_base_mesh_cache.py'), '--config', str(CONFIG), '--max-samples', '4'], 'base mesh cache')

In [ ]:
from image_to_3d_shapenet import load_config
from image_to_3d_shapenet.dataset import RenderedRefinementDataset
config = load_config(CONFIG)
dataset = RenderedRefinementDataset(PROJECT_ROOT, config, maximum=4, image_resolution=64)
batch = dataset[0]
print({'samples': len(dataset), 'model_id': batch['model_id'], 'view_id': batch['view_id'], 'rgb': tuple(batch['rgb'].shape), 'base_vertices': len(batch['base_vertices']), 'target_vertices': len(batch['target_vertices']), 'backend': batch['base_backend']})

In [ ]:
train_command = [sys.executable, str(PROJECT_ROOT / 'scripts' / 'train.py'), '--config', str(CONFIG), '--quick-overfit', '--checkpoint-dir', str(CHECKPOINT_DIR)]
run_checked(train_command, 'quick train')

In [ ]:
last_checkpoint = CHECKPOINT_DIR / 'last.pt'
assert last_checkpoint.is_file()
resume_command = [sys.executable, str(PROJECT_ROOT / 'scripts' / 'train.py'), '--config', str(CONFIG), '--quick-overfit', '--resume', str(last_checkpoint), '--checkpoint-dir', str(CHECKPOINT_DIR)]
run_checked(resume_command, 'checkpoint resume')
print('Resume checkpoint verified:', last_checkpoint)

In [ ]:
import csv
from PIL import Image
history = list(csv.DictReader((CHECKPOINT_DIR / 'training_history.csv').open(encoding='utf-8')))
print(history)
display(Image.open(CHECKPOINT_DIR / 'loss_plot.png'))
for name in ('last.pt', 'best_chamfer.pt', 'best_edge.pt', 'config_snapshot.yaml'):
    assert (CHECKPOINT_DIR / name).is_file(), name
disk = shutil.disk_usage(PROJECT_ROOT)
peak_vram = torch.cuda.max_memory_allocated() / 1024**3 if CUDA else 0.0
print({'elapsed_seconds': round(time.perf_counter() - STARTED, 2), 'peak_vram_gb': round(peak_vram, 3), 'disk_free_gb': round(disk.free / 1024**3, 2), 'resume_checkpoint': str(last_checkpoint)})